# Data Cleaning

As a Data Scientist, you will spend most of your time on data cleaning. It is a process which can be annoying sometimes and definitely one you have to do over and over and over again. But it is also one of the most important steps of the data science lifecycle since it determines the quality of your data. Remember the "garbage in, garbage out" concept! Therefore it has a huge impact on your prediction model later on and is time well spent. Most of the data you will see in this course is already pretty much cleaned up for modelling. In a business context, this is usually not the case.

**Notebook 2 of 6.** This one takes a raw Seattle weather file and makes it trustworthy. Missing values are deliberately left alone here: they get a notebook of their own.

## Learning Objectives

At the end of this notebook, you should be able to:

* Inspect a raw dataset with the general pandas functions and spot what is wrong with it
* Standardise column names, and name a column that has none
* Find and remove duplicate rows
* Correct data types, and recognise what an `object` dtype is hiding
* Fix a column stored in the wrong unit
* Reduce a messy categorical column to a controlled set of values
* Save an intermediate result for the next stage of the pipeline

> **Tip.** If you have not read `01_what_is_eda.md`, do that first. It is twenty minutes and it
> names the six things that go wrong in a dataset, which is what this notebook then fixes
> one at a time. The `README.md` lists all six lessons and the order they run in.

## Preliminary data cleaning

As it gets probably usual by now, we import our most needed libraries.

In [ ]:
import numpy as np
import pandas as pd

Let's load the data from `seattle-weather_raw.csv` into a pandas data frame. You might already know this dataset from the pandas exercises notebooks. We gave you the dataset preprocessed. Now it's the raw file and we need to give it some thought to clean it up.

In [ ]:
# importing raw data
df_weather = pd.read_csv("data/seattle-weather_raw.csv")

> **Exercise:**
>
> Get a quick overview of the data. Use the learned pandas methods like `.head()`, `.tail()`, `.info()` and `.describe()`. Is there anything you notice? What would you address in the data cleaning process?

<details><summary>
Click here for a hint…
</summary>
Do you see some:
  
    - formatting problems in the column names?
    - duplicates? 
    - missing values? (what is the reason? and what can you do about it?)
    - inconsistency in the meaning of data?
    - different unit for data?
    - different data types than expected?
    - obvious outliers/ unexpected values?
</details>

In [ ]:
# your code: check for data types and statistics overview of data frame columns

In [ ]:
df_weather.head(7)

Let's get started!

## Fix column names formats

Three things are wrong with the column names, and each of them will bite you later:

- they contain **spaces**, so `df_weather.temp max` is not something you can write.
- they mix **upper and lower case**, which is one more thing to remember correctly.
- the last one has **no name at all**, so there is no way to refer to it.

The `.str` accessor applies string methods to the column index itself, which handles the
first two. For the third, `.rename` takes a dictionary mapping the old name to the new one.

In [ ]:
# spaces to underscores, then everything to lower case
df_weather.columns = df_weather.columns.str.replace(" ", "_").str.lower()

# give the unnamed column a name
df_weather = df_weather.rename({"-": "weather"}, axis=1)

df_weather.columns

Congratulations. You just finished the first step of cleaning this data frame. Let's have another look at the first rows again. What else can you see which is odd?

In [ ]:
df_weather.head(5)

## Dealing with duplicates

The rows with index $2$ and $3$ are identical. There are obviously duplicated which can happen for example while merging two data frames together. The function `.duplicated()` returns a boolean pandas Series which tells you if a row is a duplicate or not. Combined with another well known function it will tell us how many duplicated rows we have.

In [ ]:
# check how many duplicated rows exist in the data frame
df_weather.duplicated().value_counts()

As you can see we have 37 duplicated rows. Those rows can be easily dropped. Furthermore we have to drop the old index and renew them.

In [ ]:
# remove duplicates
df_weather = df_weather.drop_duplicates()
# reset index inplace
df_weather.reset_index(inplace=True, drop=True)
df_weather.head(5)

## Data Types and Transforming Data

One Part of Data Cleaning is to get the data types into the right shape. This is important so we can work better with the data later on. The dtype `object` means that there are several data types in the corresponding column.

In [ ]:
# check data types in data frame
df_weather.dtypes

Start at the beginning, with **date**. Its dtype is `object`, which in pandas means
"something that is not a number", most often a string. Converting it to a real datetime is
what unlocks the `.dt` accessor and time based selection, and you will need both once you start exploring the data.

`pd.to_datetime` does the conversion. You could also have done it while reading the file,
with `pd.read_csv(..., parse_dates=["date"])`.

In [ ]:
# before the conversion, each entry is a plain string
print("before:", type(df_weather["date"][0]))

# convert, telling pandas the format the dates are written in
df_weather["date"] = pd.to_datetime(df_weather["date"], format="%Y/%m/%d")

print("after :", type(df_weather["date"][0]))

Moving on to the column **precipitation**. At the look of the data frames head, we would have expected a float datatype. This is actually an indication, that there is more than just numbers in this column. For now we could just try to convert the column to the float type, also this might not work.

In [ ]:
# change data type to float
df_weather = df_weather.astype({"precipitation": float})

As expected we get an error message. However this can help us to determine why our column cannot be assigned to the data type `float`. If we read the message it tells us in the last line that we can not convert a string to a float. Also now we know the cause of our error. Somewhere in this column is a `$`-character hidden. Out of curiosity let's try to find it.

In [ ]:
# display rows with "$" in precipitation column
df_weather.query('precipitation == "$"')

It is only one `$`-character in line $1130$. We can easily replace the `$`-character with a numpy nan value. nan stands for "Not a Number" and is used as a placeholder for missing values. Afterwards we are able to convert our column data type.

In [ ]:
# replace the `$`-character with a numpy NaN value
df_weather["precipitation"] = df_weather.precipitation.replace("$", np.nan)
# change data type to float
df_weather = df_weather.astype({"precipitation": float})
df_weather.precipitation.dtypes

Looking at our data types and the table head again, the `temp_max`-column looks good. But if we continue with `temp_min` we will notice two things. First there are not only numbers in `temp_min` (dtype is detected as object) and the minimum temperature is larger as the maximum temperature. We can assume that somehow the units of the temperature got mixed up. The maximum temperature is in Celsius while the minimum is appearing to be in Fahrenheit. To fix that we can first delete the unit from the number and change the data type.

In [ ]:
# remove unit from "temp_min" column, change dtype to float
df_weather["temp_min"] = df_weather.temp_min.str.strip(" F").astype("float")
df_weather.temp_min.dtype

Now that we have floats in our column we can convert the temperature from Fahrenheit to Celsius.

In [ ]:
# convert temp_min to celsius
df_weather["temp_min"] = df_weather.temp_min.apply(lambda x: (x - 32) / 1.8)
df_weather.head(5)

That is much better. Also the minimum temperature is now smaller than the maximum temperature.

For the "Wind" column, the data type is already as it should be. However, you should round these numbers for the future. It might be good to have a few decimal places for future modelling, but it is not always necessary to be so precise. Let's therefore try rounding these numbers for practice purposes.

In [ ]:
# rounding wind data
df_weather["wind"] = df_weather.wind.round(2)
df_weather.head(5)

The last column holds strings describing the type of weather. That makes it a
**categorical** variable, and the first two questions to ask about one are always how many
distinct values it has, and what they are.

In [ ]:
# how many distinct weather values are there, and which ones
print("distinct values:", df_weather.weather.nunique())
df_weather["weather"].unique()

Looking at the unique values, we see that "drizzle", "d", "d." and "drizzle" all probably represent the same category. So one category has several values. This often happens if there is no fixed standard for data collection. We need to reduce the number of values for each category to one. The easiest and safest way is to manually assign each value to the appropriate category. The implemented function `.map` uses either a dictionary to assign a value to it's new one or a function, which is applied on each value. In our case we will use the dictionary.

> **Note.** Sometimes it can happen that abbreviations are ambiguous. For example, there is a category "sn" which could represent either "snow" or "sun". Here it is worth diving deeper into the data collection process to make a more informed decision.

> You could also use pattern match (which uses regular expressions) or fuzzy matching to approach this task. But they are more complicated to program and an absolute understanding of them is mandatory, because there is a risk of assigning a value to the wrong category.

In [ ]:
# reducing number of categorical values by mapping with a word-dictionary
df_weather["weather"] = df_weather["weather"].map(
    {
        "drizzle": "drizzle",
        "r": "rain",
        "r.": "rain",
        "rain": "rain",
        "s": "sun",
        "Rain": "rain",
        "sun": "sun",
        "Sun": "sun",
        "Snow": "snow",
        "sn.": "snow",
        "sw": "snow",
        "d": "drizzle",
        "s.": "sun",
        "driz.": "drizzle",
        "snow": "snow",
        "Drizzle": "drizzle",
        "Fog": "fog",
        "f": "fog",
        "f.": "fog",
        "fog": "fog",
    }
)
df_weather.weather.unique()

We can convert the data type from this column to `string` just as easily as before. Note that this is not mandatory going forward.

In [ ]:
# convert to dtype "string"
df_weather["weather"] = df_weather.weather.astype("string")
df_weather.weather.dtype

## Saving your work

The next notebook starts from the data frame you have just cleaned, so write it out.
Notice that the missing values are **not** filled in here. Deciding what to do with them
is the whole of `03_missing_data.ipynb`, and that decision deserves its own notebook.

In [ ]:
# hand the cleaned data to the next notebook.
# missing values are deliberately left in: 02_missing_data is where they are dealt with.
df_weather.to_csv("data/seattle-weather_clean.csv", index=False)

print(f"saved {df_weather.shape[0]} rows and {df_weather.shape[1]} columns")